## Real-Time Bitcoin Price Analysis using Docker SDK for Python

This notebook demonstrates how to use the Docker SDK for Python to build and run a fully automated, containerized real-time Bitcoin analytics pipeline. It includes:

- Automated building and launching of Docker containers for InfluxDB, Grafana, and a BTC fetcher
- Ingestion of both historical and real-time BTC/USD price data from the CryptoCompare API
- Visualization of real-time and historical data in a Grafana dashboard with source-tag filtering

## Imports

We import our utility functions (which wrap the Docker SDK and BTC pipeline logic) along with standard Python libraries.

In [ ]:
# Cell 1: Imports
import docker_sdk_utils as utils
import os
import pandas as pd
import time

### Start the pipeline using Docker SDK

This step:
- Builds the Docker image for the BTC fetcher (if not already built)
- Starts containers for InfluxDB and Grafana
- Starts the BTC fetcher container, which fetches 60 historical points and 1 real-time point per cycle

### Step 1: Start InfluxDB Container

This container runs the InfluxDB time-series database to store real-time Bitcoin price data. The container is launched using Docker SDK.

In [ ]:
influxdb_container = utils.start_influxdb_container(container_name='influxdb', port=8086)
print("Started InfluxDB container.")
time.sleep(10)  # Wait for InfluxDB to fully initialize

INFO  > cmd='/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-783e0930-1631-4d64-8bb4-f3a98bb74fcd.json'


### Step 2: Start Grafana Container

Grafana provides live dashboards to visualize the Bitcoin price data stored in InfluxDB. This step also ensures the dashboard and data source are auto-provisioned.

In [ ]:
grafana_container = utils.start_grafana_container(container_name='grafana', port=3000)
print("Started Grafana container (open http://localhost:3000, default login admin/admin).")
time.sleep(5)

### View Your Real-Time Dashboard

Visit [http://localhost:3000](http://localhost:3000) and log in to Grafana.

- Explore the `BTC/USD Close Price` graph to view historical trends.
- Use the `source` dropdown to toggle between `historical` and `realtime`.
- Check the `Current BTC/USD Price` stat panel for live updates every 10s.

## Set InfluxDB Configuration

We use environment variables for security, but you can hardcode values if running locally.

In [ ]:
INFLUXDB_URL = "http://localhost:8086"
INFLUXDB_TOKEN = os.getenv("INFLUXDB_ADMIN_TOKEN")
INFLUXDB_ORG = os.getenv("INFLUXDB_ORG")
INFLUXDB_BUCKET = os.getenv("INFLUXDB_BUCKET")

In [ ]:
result, logs = utils.start_btc_fetcher_container(
    influxdb_url=INFLUXDB_URL,
    influxdb_token=INFLUXDB_TOKEN,
    influxdb_org=INFLUXDB_ORG,
    influxdb_bucket=INFLUXDB_BUCKET,
    container_name='btc-fetcher'
)
print("BTC fetcher job logs:\n", logs)

### Step 3: Start Real-Time BTC Fetcher

This container fetches the current BTC/USD price every 10 seconds using the CryptoCompare API and writes the data to InfluxDB.

In [ ]:
fetcher = utils.CryptoDataFetcher(
    influxdb_url=INFLUXDB_URL,
    influxdb_token=INFLUXDB_TOKEN,
    influxdb_org=INFLUXDB_ORG,
    influxdb_bucket=INFLUXDB_BUCKET
)
data = fetcher.query_data(start="-30m")
df = pd.DataFrame(data)
close_prices = df[df['field']=='close'][['time', 'value']].sort_values('time')
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.plot(close_prices['time'], close_prices['value'], label='Close Price')
plt.title('BTC/USD Close Price (Last 30 min)')
plt.xlabel('Time')
plt.ylabel('Price')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig("btc_close_price.png")  # Save instead of show
print("Plot saved as btc_close_price.png")

### Step 4: Run Analysis Container

This container runs a Python script that queries the latest data from InfluxDB, applies a moving average and ARIMA model, and saves a plot to disk.

In [ ]:
# Time Series Analysis
ts_df = fetcher.time_series_analysis(close_prices.reset_index(drop=True), order=(1,1,1), window=5)
plt.figure(figsize=(10,4))
plt.plot(ts_df['time'], ts_df['value'], label='Close Price')
plt.plot(ts_df['time'], ts_df['moving_avg'], label='Moving Avg (5)')
if ts_df['arima_forecast'].notnull().any():
    plt.plot(ts_df['time'], ts_df['arima_forecast'], label='ARIMA Forecast', linestyle='--')
plt.title('BTC/USD Close Price: Moving Avg & ARIMA')
plt.xlabel('Time')
plt.ylabel('Price')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig("btc_analysis.png")  # Save instead of show
print("Analysis plot saved as btc_analysis.png")

### Step 5: Visualize Analysis Output

The following cell displays the generated plot (`btc_analysis.png`) created by the analysis container.

In [ ]:
from IPython.display import Image
Image(filename="output/btc_analysis.png")

### Stop and Clean Up Containers

This step gracefully stops and removes all running containers, including:
- `btc-fetcher`
- `influxdb`
- `grafana`

This ensures a clean environment for the next run.

In [ ]:
utils.stop_docker_container('influxdb')
utils.stop_grafana_container('grafana')

# Summary

This notebook showed how to build a fully containerized, reproducible pipeline for real-time financial data ingestion and analysis, leveraging the Docker SDK for Python.

**Next steps:**  
- Schedule automatic data pulls
- Add Grafana for real-time dashboards
- Support multiple cryptocurrencies or more advanced analytics